# Fase 0 — Extracción de datos de Firestore

Este notebook extrae la jerarquía completa de Firestore (`sessions → levels → rooms`)
y la guarda en `data/raw/` como JSON estructurado.

**Prerrequisitos:**
- `credentials/firebase-service-account.json` presente (no subir a git)
- `pip install cryptography` (única dependencia extra)

**Salidas:**
- `data/raw/sessions_raw.json` — jerarquía completa sin modificar
- `data/raw/extraction_metadata.json` — metadatos de la extracción (fecha, conteos)

## 0. Configuración

In [ ]:
import sys
import json
import datetime
from pathlib import Path

# Añadir src/ al path para importar firestore_client
sys.path.insert(0, str(Path('..') / 'src'))

from firestore_client import FirestoreClient

# Rutas
CREDENTIALS_PATH = Path('..') / 'credentials' / 'firebase-service-account.json'
RAW_DIR          = Path('..') / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f'Credenciales: {CREDENTIALS_PATH.resolve()}')
print(f'Salida raw:   {RAW_DIR.resolve()}')

## 1. Conexión y extracción

In [ ]:
client = FirestoreClient(str(CREDENTIALS_PATH))
print(f'Proyecto Firebase: {client.get_project_id()}')

In [ ]:
print('Extrayendo sesiones de Firestore...')
sessions = client.export_all_sessions()
print(f'\n✓ Total sesiones extraídas: {len(sessions)}')

## 2. Resumen de lo extraído

In [ ]:
total_levels = sum(len(s['levels']) for s in sessions)
total_rooms  = sum(len(l['rooms']) for s in sessions for l in s['levels'])

print(f'Sesiones : {len(sessions)}')
print(f'Niveles  : {total_levels}')
print(f'Salas    : {total_rooms}')

# Versiones y plataformas presentes
versions  = sorted({s.get('gameVersion', '?') for s in sessions})
platforms = sorted({s.get('platform', '?') for s in sessions})
elements  = sorted({s.get('playerElement', '?') for s in sessions})

print(f'\nVersiones de juego : {versions}')
print(f'Plataformas        : {platforms}')
print(f'Elementos jugados  : {elements}')

victories = sum(1 for s in sessions if s.get('isVictory'))
print(f'\nVictorias: {victories}/{len(sessions)} ({victories/len(sessions)*100:.1f}%)')

## 3. Inspección de una sesión

In [ ]:
# Primera sesión — vista completa
s0 = sessions[0]
print(f'--- Sesión: {s0["sessionId"]} ---')
print(f'Elemento  : {s0.get("playerElement")}')
print(f'Victoria  : {s0.get("isVictory")}')
print(f'Kills     : {s0.get("totalKills")}')
print(f'Muertes   : {s0.get("totalDeaths")}')
print(f'Tiempo(s) : {s0.get("totalTimeSecs")}')
print(f'Niveles completados: {s0.get("levelsCompleted")}')
print(f'Comentario: {s0.get("playerComment") or "(vacío)"}')

print(f'\nNiveles: {len(s0["levels"])}')
for lv in s0['levels']:
    print(f'  {lv["levelId"]} — kills:{lv.get("kills")} deaths:{lv.get("deaths")} time:{lv.get("timeSecs")}s | {len(lv["rooms"])} salas')
    for rm in lv['rooms']:
        dynamic = {k: v for k, v in rm.items() if k not in ('roomId','timeSecs','deaths','damageTaken','firstSpell')}
        print(f'    {rm["roomId"]} | firstSpell:{rm.get("firstSpell")} | {dynamic}')

## 4. Guardar en data/raw/

In [ ]:
# Guardar jerarquía completa
output_path = RAW_DIR / 'sessions_raw.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(sessions, f, ensure_ascii=False, indent=2, default=str)

print(f'✓ Datos guardados en {output_path}')
print(f'  Tamaño: {output_path.stat().st_size / 1024:.1f} KB')

In [ ]:
# Guardar metadatos de extracción
metadata = {
    'extracted_at'   : datetime.datetime.utcnow().isoformat() + 'Z',
    'project_id'     : client.get_project_id(),
    'total_sessions' : len(sessions),
    'total_levels'   : total_levels,
    'total_rooms'    : total_rooms,
    'game_versions'  : versions,
    'platforms'      : platforms,
    'elements'       : elements,
    'victories'      : victories,
    'defeat_rate'    : round(1 - victories / len(sessions), 4) if sessions else None,
}

meta_path = RAW_DIR / 'extraction_metadata.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f'✓ Metadatos guardados en {meta_path}')
print(json.dumps(metadata, indent=2))

## 5. Inventario de campos dinámicos

Los campos de sala son dinámicos (dependen de qué enemigos/hechizos aparecen).
Este bloque los cataloga para planificar el preprocessing.

In [ ]:
from collections import defaultdict

# Agrupar campos de sala por prefijo
prefix_counts = defaultdict(set)
static_fields = {'roomId', 'levelId', 'sessionId', 'timeSecs', 'deaths', 'damageTaken', 'firstSpell'}

for s in sessions:
    for lv in s['levels']:
        for rm in lv['rooms']:
            for k in rm:
                if k in static_fields:
                    continue
                prefix = k.split('_')[0] if '_' in k else k
                suffix = k[len(prefix)+1:] if '_' in k else ''
                prefix_counts[prefix].add(suffix)

print('Prefijos de campos dinámicos en salas:')
for prefix, values in sorted(prefix_counts.items()):
    print(f'  {prefix}_* → {sorted(values)[:8]}{" ..." if len(values) > 8 else ""}')

---
**Siguiente paso:** `02_preprocessing.ipynb` — aplanar la jerarquía en DataFrames.